In [ ]:
import numpy as np
import pandas as pd
import emoji
import re
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import glob

In [4]:
raw_csv_folder = Path('data/raw')

In [6]:
csv_files = glob.glob(os.path.join(raw_csv_folder, "*.csv"))
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

In [7]:
df.shape

(255708, 8)

In [8]:
df.head()

,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,19d73b7cd0a23f961ea3aae6d1554c62b004aac1a4a6c7...,2025-12-24 05:22:29.307373+00:00,en,sandiegouniontribune.com,sandiegouniontribune.com,2025-12-23,UC San Diego eyes campuswide AI coordinating s...,https://www.sandiegouniontribune.com/2025/12/2...
1,8ea4c1650576763e5fe3471911fbdbacaf7edafc3ab052...,2025-12-24 05:20:33.347048+00:00,en,townhall.com,townhall.com,2025-12-23,President Trump Takes a Victory Lap Over Fanta...,https://townhall.com/tipsheet/amy-curtis/2025/...
2,fbebfda1017f09254d799296d2d28dc6d4306b68c432b6...,2025-12-24 04:25:36.871520+00:00,en,dailycaller.com,dailycaller.com,2025-12-23,Kevin Hassett Assesses New Numbers And Explain...,https://dailycaller.com/2025/12/23/kevin-hasse...
3,5dc81d658b56b2424da775f9d2496810172b4e7e907305...,2025-12-24 03:21:38.495827+00:00,en,theweek.com,theweek.com,2025-12-23,Trump vs. states: Who gets to regulate AI?,https://theweek.com/politics/trump-states-regu...
4,8c34961a3af69294198033d38a21cca091baf4c89907ab...,2025-12-24 03:21:19.174848+00:00,en,theweek.com,theweek.com,2025-12-23,Tariffs have American whiskey distillers on th...,https://theweek.com/business/economy-whiskey-t...


In [9]:
keywords = ["ai","a.i.","artificial intelligence","deepfake!","deep fake!","chatgpt","chat GPT","chatbot!","language model!","machine learning"]
pattern = r'\b(?:' + '|'.join(keywords) + r')\b'
final_df = df[df['title'].str.contains(pattern,case=False,na=False)]
final_df.shape

(61362, 8)

In [10]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 61362 entries, 0 to 255707
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            61362 non-null  str  
 1   indexed_date  61362 non-null  str  
 2   language      61362 non-null  str  
 3   media_name    61362 non-null  str  
 4   media_url     61362 non-null  str  
 5   publish_date  61362 non-null  str  
 6   title         61362 non-null  str  
 7   url           61362 non-null  str  
dtypes: str(8)
memory usage: 4.2 MB


In [ ]:
final_df.to_csv("AI_articles_metadata.csv")

In [ ]:
data_df = pd.read_csv('../data/articles/AI_articles_metadata.csv')

In [27]:
from newspaper import Article, Config
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import requests

In [28]:
config = Config()
config.fetch_images = True
config.memoize_articles = False

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
    'Cache-Control': 'max-age=0',
    'Referer': 'https://www.google.com/',
}

def scrape_article(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        article = Article(url, config=config)
        article.set_html(response.text)
        article.parse()

        return {
            "url": url,
            "title": article.title,
            "text": article.text,
            "top_image": article.top_image,
            "images": list(article.images)
        }

    except Exception as e:
        return {
            "url": url,
            "error": str(e)
        }

In [7]:
def scrape_bulk(urls, max_workers=24):
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(scrape_article, url) for url in urls]

        for future in tqdm(as_completed(futures), total=len(futures)):
            results.append(future.result())

    return results

In [9]:
urls = data_df['url']

In [10]:
results = scrape_bulk(urls, max_workers=24)

100%|██████████| 61362/61362 [49:15<00:00, 20.76it/s]  


In [11]:
article_df = pd.DataFrame(results)

In [14]:
article_df.head(5)

,url,error,title,text,top_image,images
0,https://time.com/7341580/christian-ai-backlash/,Article `download()` failed with 406 Client Er...,NaN,NaN,NaN,NaN
1,https://www.reuters.com/business/media-telecom...,Article `download()` failed with 401 Client Er...,NaN,NaN,NaN,NaN
2,https://www.theguardian.com/technology/2025/de...,NaN,"Elon Musk, AI and the antichrist: the biggest ...","Hello, and welcome to TechScape. I’m your host...",https://i.guim.co.uk/img/media/deb2b33ba7ff621...,[https://i.guim.co.uk/img/media/deb2b33ba7ff62...
3,https://theconversation.com/how-can-canada-bec...,NaN,How can Canada become a global AI powerhouse? ...,Artificial intelligence is everywhere. In fact...,https://images.theconversation.com/files/70923...,"[data:image/gif;base64,R0lGODlhAQABAAD/ACwAAAA..."
4,https://www.newsweek.com/nw-ai/ai-impact-is-ai...,Article `download()` failed with 406 Client Er...,NaN,NaN,NaN,NaN


In [15]:
article_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61362 entries, 0 to 61361
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        61362 non-null  str   
 1   error      26613 non-null  str   
 2   title      34749 non-null  str   
 3   text       34749 non-null  str   
 4   top_image  34749 non-null  str   
 5   images     34749 non-null  object
dtypes: object(1), str(5)
memory usage: 2.8+ MB


In [16]:
article_df.to_csv('Data1.csv')

In [17]:
failed_df = article_df[article_df["error"].notna()]

In [51]:
success_df1 = article_df[article_df['error'].isna()]

In [52]:
success_df1.shape

(34749, 6)

In [18]:
failed_df.shape

(26613, 6)

In [19]:
failed_df

,url,error,title,text,top_image,images
0,https://time.com/7341580/christian-ai-backlash/,Article `download()` failed with 406 Client Er...,NaN,NaN,NaN,NaN
1,https://www.reuters.com/business/media-telecom...,Article `download()` failed with 401 Client Er...,NaN,NaN,NaN,NaN
4,https://www.newsweek.com/nw-ai/ai-impact-is-ai...,Article `download()` failed with 406 Client Er...,NaN,NaN,NaN,NaN
7,https://www.reuters.com/legal/litigation/ai-co...,Article `download()` failed with 401 Client Er...,NaN,NaN,NaN,NaN
13,https://www.marketwatch.com/story/servicenow-a...,Article `download()` failed with 401 Client Er...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
61355,https://www.benzinga.com/pressreleases/25/05/g...,Article `download()` failed with 405 Client Er...,NaN,NaN,NaN,NaN
61356,https://www.benzinga.com/tech/25/05/45125414/m...,Article `download()` failed with 405 Client Er...,NaN,NaN,NaN,NaN
61358,https://www.benzinga.com/pressreleases/25/05/n...,Article `download()` failed with 405 Client Er...,NaN,NaN,NaN,NaN
61359,https://www.benzinga.com/pressreleases/25/05/n...,Article `download()` failed with 405 Client Er...,NaN,NaN,NaN,NaN


In [30]:
failed_urls = failed_df['url']
failed_url_result = scrape_bulk(failed_urls, max_workers=24)

100%|██████████| 26613/26613 [26:24<00:00, 16.79it/s] 


In [31]:
article_df2 = pd.DataFrame(failed_url_result)

In [32]:
article_df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 26613 entries, 0 to 26612
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        26613 non-null  str   
 1   error      25042 non-null  str   
 2   title      1571 non-null   str   
 3   text       1571 non-null   str   
 4   top_image  1571 non-null   str   
 5   images     1571 non-null   object
dtypes: object(1), str(5)
memory usage: 1.2+ MB


In [33]:
article_df2.head()

,url,error,title,text,top_image,images
0,https://time.com/7340901/ai-history-bubble-ben...,406 Client Error: Not Acceptable for url: http...,NaN,NaN,NaN,NaN
1,https://time.com/7341580/christian-ai-backlash/,406 Client Error: Not Acceptable for url: http...,NaN,NaN,NaN,NaN
2,https://www.marketwatch.com/story/servicenow-a...,NaN,ServiceNow to buy Armis for $7.75 billion as i...,The acquisition of cybersecurity company Armis...,https://images.mktw.net/im-56113489/social,"[data:image/svg+xml;base64,PHN2ZwogIGZvY3VzYWJ..."
3,https://www.newsweek.com/nw-ai/ai-impact-is-ai...,406 Client Error: Not Acceptable for url: http...,NaN,NaN,NaN,NaN
4,https://www.bloomberg.com/opinion/articles/202...,403 Client Error: Forbidden for url: https://w...,NaN,NaN,NaN,NaN


In [45]:
article_df2['error'][0]

'406 Client Error: Not Acceptable for url: https://time.com/7340901/ai-history-bubble-benchmarks/'

In [42]:
article_df2['error'].value_counts()[:5]

error
All strings must be XML compatible: Unicode or ASCII, no NULL bytes or control characters                                                                                    1184
HTTPSConnectionPool(host='www.usnews.com', port=443): Read timed out. (read timeout=10)                                                                                        42
HTTPSConnectionPool(host='www.newsmax.com', port=443): Read timed out. (read timeout=10)                                                                                       21
HTTPSConnectionPool(host='www.benzinga.com', port=443): Read timed out. (read timeout=10)                                                                                       4
404 Client Error: Not Found for url: https://www.inquirer.com/wires/ap/tech-stocks-tank-chinese-competitor-threatens-upend-ai-frenzy-nvidia-sinks-nearly-17-20250127.html       4
Name: count, dtype: int64

In [41]:
article_df2['error'].nunique()

23731

In [53]:
success_df2 = article_df2[article_df2["error"].isna()]

In [54]:
success_df2.head()

,url,error,title,text,top_image,images
2,https://www.marketwatch.com/story/servicenow-a...,NaN,ServiceNow to buy Armis for $7.75 billion as i...,The acquisition of cybersecurity company Armis...,https://images.mktw.net/im-56113489/social,"[data:image/svg+xml;base64,PHN2ZwogIGZvY3VzYWJ..."
7,https://www.reuters.com/legal/litigation/ai-co...,NaN,AI companions meet the law: New York and Calif...,"December 23, 2025 - For years, AI ""companions""...",https://www.reuters.com/resizer/v2/QKGF3DALJVM...,[https://www.reuters.com/resizer/v2/QKGF3DALJV...
8,https://www.reuters.com/business/global-market...,NaN,AI spending spree drives global tech debt issu...,Dec 22 (Reuters) - Global technology companies...,https://www.reuters.com/resizer/v2/INH3QDQ7XZI...,[https://www.reuters.com/graphics/GLOBAL-TECH/...
9,https://www.barrons.com/articles/microsoft-sto...,NaN,"Microsoft Stock Has 29% Upside in 2026, Says D...","This copy is for your personal, non-commercial...",https://images.barrons.com/im-78168239/social,[https://www.barrons.com/asset/barrons/images/...
10,https://www.reuters.com/world/asia-pacific/byt...,NaN,ByteDance plans to spend $23 billion towards A...,Dec 22 (Reuters) - TikTok owner ByteDance has ...,https://www.reuters.com/resizer/v2/S5KUZ6227FN...,[https://www.reuters.com/resizer/v2/4GJGPOWTJ5...


In [55]:
final_article_df = pd.concat([success_df1, success_df2], ignore_index=True)

In [56]:
final_article_df.shape

(36320, 6)

In [57]:
round((final_article_df.shape[0] / data_df.shape[0])*100,2)

59.19

In [58]:
#Hence, 60% of the urls could be scraped

In [59]:
final_article_df.to_csv('AI_articles2025-2026.csv')